In [ ]:
# calculating lnL for u: code from Claude AI
# Likelihood project
# original full code here, modified for project

import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

# ======================================================================
# 1. INPUT DATA
# ======================================================================
msa = """>Bos
AGCTCAAGCTCAAGCTCAAGCTCAAGCTCA
>Ovis
CGTTACCGTTACCGTTACCGTTACCGTTAC
>Felis
CGCTACCGCTACCGCTACCGCTACCGCTAC
>Homo
AGATCAAGATCAAGATCAAGATCAAGATCA
>Mus
CGCTATCGCTATCGCTATCGCTATCGCTAT"""

# Each edge is (parent, child). Internal nodes are ints, leaves are names.
edges = [(9, "Bos"), (9, "Ovis"), (8, 9), (8, "Felis"),
         (7, "Homo"), (7, "Mus"), (6, 7), (6, 8)]

# The i-th length belongs to the i-th edge.
lengths = [0.1, 0.4, 0.01, 0.04, 0.2, 0.08, 0.12, 0.14]


# ======================================================================
# 2. ALIGNMENT CLASS (stores names, sequences, and the one-hot vectors)
# ======================================================================
class Alignment:
    STATES = "ACGT"                                   # order of the 4 vector entries
    STATE_INDEX = {s: i for i, s in enumerate(STATES)}

    def __init__(self, msa_string):
        # --- your original parsing code, unchanged in logic ---
        list_msa = msa_string.split("\n")
        self.seqDict = {}
        index = 0
        while index < len(list_msa):
            elem = list_msa[index]
            if elem[0] == ">":                        # a name line
                name = elem.lstrip(">")
                seq = list_msa[index + 1]             # sequence is on the next line
                self.seqDict[name] = seq
            index += 1

        self.names = list(self.seqDict.keys())
        self.n_sites = len(self.seqDict[self.names[0]])

        # Every sequence in an alignment must have the same length.
        assert all(len(s) == self.n_sites for s in self.seqDict.values()), \
            "Sequences have different lengths, so this is not an alignment"

        # Build the one-hot vectors ONCE for all leaves and all sites.
        # leaf_vectors[name] has shape (n_sites, 4), and row i is the state
        # vector u of that leaf at site i, e.g. A -> [1, 0, 0, 0].
        self.leaf_vectors = {name: self._to_vectors(seq)
                             for name, seq in self.seqDict.items()}

    def _to_vectors(self, seq):
        U = np.zeros((len(seq), 4))
        for i, base in enumerate(seq.upper()):
            if base in self.STATE_INDEX:
                U[i, self.STATE_INDEX[base]] = 1.0
            else:
                U[i, :] = 1.0                         # gap or N: any state possible
        return U


# ======================================================================
# 3. TREE CLASS (stores the topology and branch lengths)
# ======================================================================
class Tree:
    def __init__(self, edges, lengths):
        assert len(edges) == len(lengths), "Need one length per edge"

        # children[parent] = list of (child, branch_length)
        self.children = {}
        # branch_length[child] = length of the branch leading INTO that child.
        # Every node except the root has exactly one parent, so this is unambiguous.
        self.branch_length = {}

        all_children = set()
        for (parent, child), t in zip(edges, lengths):
            self.children.setdefault(parent, []).append((child, t))
            self.branch_length[child] = t
            all_children.add(child)

        # The root is the only parent that is never a child.
        self.root = (set(self.children) - all_children).pop()

        # Internal nodes ordered so that children come BEFORE their parents.
        # This is the order in which pruning must be performed.
        self.postorder = self._postorder(self.root)

    def _postorder(self, node):
        if node not in self.children:                 # leaf: nothing to compute
            return []
        order = []
        for child, _ in self.children[node]:
            order += self._postorder(child)
        order.append(node)                            # node comes after its children
        return order


# ======================================================================
# 4. MODEL: Jukes-Cantor Q matrix (as in your slide)
# ======================================================================
def make_Q(mu):
    """Off-diagonal entries = mu, diagonal = -3*mu, so each row sums to 0."""
    Q = np.full((4, 4), mu)
    np.fill_diagonal(Q, -3.0 * mu)
    return Q

PI = np.full(4, 0.25)                                 # equilibrium frequencies


# ======================================================================
# 5. LOG-LIKELIHOOD FOR A FIXED mu  (task 2a)
# ======================================================================
def log_likelihood(tree, aln, mu, return_sites=False):
    Q = make_Q(mu)

    # Transition matrix e^(Q*t) for each branch, computed ONCE per mu
    # (not once per site). P[child][i, j] = probability that the parent is in
    # state i and the child is in state j after time t.
    P = {child: expm(Q * t) for child, t in tree.branch_length.items()}

    site_logliks = np.zeros(aln.n_sites)

    # ---- loop over alignment columns (sites) ----
    for site in range(aln.n_sites):

        # v[node] is the 4-element likelihood vector at that node.
        # For leaves it is the one-hot vector u of the observed nucleotide.
        v = {leaf: aln.leaf_vectors[leaf][site] for leaf in aln.names}

        # Visit internal nodes from the tips toward the root (Felsenstein pruning).
        for node in tree.postorder:
            v_node = np.ones(4)                       # neutral element of the product
            for child, t in tree.children[node]:
                # Propagate the child's vector up its branch: e^(Q t) x v_child.
                # Entry i = P(subtree data below child | node has state i).
                propagated = P[child] @ v[child]
                # Merge siblings with the element-wise (Hadamard) product.
                v_node = v_node * propagated
            v[node] = v_node

        # At the root, weight by equilibrium frequencies: L(D_i | T) = pi . v_root
        site_lik = PI @ v[tree.root]
        site_logliks[site] = np.log(site_lik)

    total = site_logliks.sum()                        # sum of ln L over all sites
    return (total, site_logliks) if return_sites else total


# ======================================================================
# 6. RUN IT
# ======================================================================
aln = Alignment(msa)
tree = Tree(edges, lengths)

print("Number of sequences:", len(aln.seqDict))
print("Number of sites    :", aln.n_sites)
print("Root node          :", tree.root)
print("Postorder (internal):", tree.postorder)

# Sanity check: rows of P(t) must sum to 1 (valid probabilities).
assert np.allclose(expm(make_Q(0.3) * 0.1).sum(axis=1), 1.0)

# --- 2a: fixed mu = 0.3 ---
lnL, per_site = log_likelihood(tree, aln, mu=0.3, return_sites=True)
print(f"\nlnL at mu = 0.3: {lnL:.4f}")
print("First 5 per-site ln-likelihoods:", np.round(per_site[:5], 4))

# --- 2b: range of mu values ---
mu_values = np.linspace(0.02, 2.0, 100)
lnL_values = [log_likelihood(tree, aln, mu) for mu in mu_values]

# Coarse estimate from the grid ...
mu_hat_grid = mu_values[np.argmax(lnL_values)]
# ... refined with a 1-D optimiser (maximise lnL = minimise -lnL).
res = minimize_scalar(lambda m: -log_likelihood(tree, aln, m),
                      bounds=(0.001, 5.0), method="bounded")
mu_hat = res.x
print(f"mu-hat (grid)     : {mu_hat_grid:.3f}")
print(f"mu-hat (optimised): {mu_hat:.4f}   lnL = {-res.fun:.4f}")

# --- plot like the slide ---
plt.plot(mu_values, lnL_values, color="navy")
plt.axvline(mu_hat, color="firebrick", linestyle="--", label=f"µ̂ = {mu_hat:.3f}")
plt.xlabel("µ")
plt.ylabel("ln L")
plt.title("Log-likelihood as a function of µ")
plt.legend()
plt.show()
